### Structured  Output
Models can be requested to provide their response in a format matching a given schema.this is useful for ensuring  the output can easily parsed and used in  subsquent processing .langchain supports multiple schema types and methods  for enforcing structured output.


### Pydantic
Pydantic model proivide the richest feature set with field validation ,description,and nested Structures.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()
model = ChatGroq(model="qwen/qwen3-32b")

print(os.getenv("GROQ_API_KEY"))  # optional check

In [ ]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The tittle of the movie")
    year:int=Field(description="The year the movie was released")
    Director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movie rating out of 10")

In [ ]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

In [ ]:
model.invoke("provide the details about the movie Inception")

In [ ]:
model_with_structure.invoke("provide the details about the movie Inception")

## Message output Alongside Parsed Structure

In [ ]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    """ A movie with details"""
    title:str=Field(description="The tittle of the movie")
    year:int=Field(description="The year the movie was released")
    Director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movie rating out of 10")

model_with_structure=model.with_structured_output(Movie,include_raw=True)    

response = model_with_structure.invoke("provide the details about the movie Inception")
response

### Nested structure

In [ ]:
from pydantic import BaseModel,Field

class Actor(BaseModel):
    name:str
    role:str
    
class MovieDetails(BaseModel):
    title :str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None = Field(None,description="Budget in millons USD")    

In [ ]:
model_with_structure=model.with_structured_output(MovieDetails)    

response = model_with_structure.invoke("provide the details about the movie Inception")
response

#### TypeDict
TypeDict provides a simpler alternative using Python's built-in typing,ideal when you don't need runtime validation

In [ ]:
from typing_extensions import TypedDict,Annotated

In [ ]:
class MovieDict(TypedDict):
    """ A movie with details"""
    title:Annotated[str, ...,"The tittle of the movie"]
    year:Annotated[int, ..., "The year the movie was released"]
    Director:Annotated[str, ..., "The director of the movie"]
    rating:Annotated[float, ..., "The movie rating out of 10"]


In [ ]:
model_withtypedict=model.with_structured_output(MovieDict)    

response = model_withtypedict.invoke("provide the details about the movie Bahubali")
response

In [ ]:

class Actor(TypedDict):
    name:str
    role:str
    
class MovieDetails(TypedDict):
    title :str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None = Field(None,description="Budget in millons USD")  

model_with_structure=model.with_structured_output(MovieDetails)    

response = model_with_structure.invoke("provide the details about the movie Bahubali")
response

In [ ]:
model.profile

### Data Classes

A data class is a class typically containing mainly data,although there aren't really any restrictions.You creste it using the @dataclass decorator

In [ ]:
from pydantic import BaseModel, Field 
from langchain.agents import create_agent

class ContactInfo(BaseModel):
      """Contact information for a person."""
      name: str =Field(description="The name of the person")
      email: str = Field(description="The email address of the person")
      phone: str = Field(description="The phone number of the person")

agent = create_agent(
     model="groq:llama-3.1-8b-instant",
     response_format=ContactInfo # Auto-selects ProviderStrategy
) 

result =agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]

 })
result

In [ ]:
result['structured_response']

In [ ]:
## TypeDict
from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
      """Contact information for a person."""
      name: str 
      email: str 
      phone: str

agent = create_agent(
     model="groq:llama-3.1-8b-instant",
     response_format=ContactInfo # Auto-selects ProviderStrategy
) 

result =agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]

 })
result["structured_response"]

In [ ]:
## dataclass
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
      """Contact information for a person."""
      name: str 
      email: str 
      phone: str

agent = create_agent(
     model="groq:llama-3.1-8b-instant",
     response_format=ContactInfo # Auto-selects ProviderStrategy
) 

result =agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]

 })
result["structured_response"]